# FuseLens: Long-Context Genomic Foundation Models for DNA Breakpoint Detection

This notebook implements the complete **FuseLens** framework, a deep learning pipeline designed to detect and localize gene fusion breakpoints with high precision.

### Scientific Context
Traditional alignment-based tools often fail in repetitive genomic regions. **FuseLens** overcomes this by treating DNA as a language. We fine-tune **HyenaDNA**, a genomic foundation model capable of processing **20kb context windows**, to distinguish true oncogenic fusions from biological noise.

### Key Capabilities
1.  **Hard-Negative Data Engineering:** Integration of 5 distinct data sources, including synthetic "Hard Negatives" (Reversed/Shuffled sequences) to force the model to learn structural syntax.
2.  **HyenaDNA Backbone:** Uses sub-quadratic long convolutions to process sequences 40x longer than standard BERT models.
3.  **Attention Pooling & Localization:** A custom head that not only classifies the sequence but **localizes the specific breakpoint** by extracting attention weights.
4.  **Robust Inference:** Confidence scoring and automatic thresholding for clinical reporting.

---

## 1. Installation & Environment Setup

Installing dependencies for Hugging Face Transformers, PyTorch, and data processing.

In [ ]:
# Install required packages
!pip install -q torch transformers accelerate pandas scikit-learn matplotlib seaborn safetensors

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, roc_curve, auc
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
from safetensors.torch import load_model, save_model
from typing import Dict, List, Tuple
from tqdm.auto import tqdm
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Create output directory
if not os.path.exists("./hyenadna_breakpoint_model"):
    os.makedirs("./hyenadna_breakpoint_model", exist_ok=True)

# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup complete. Using device: {device}")

## 2. Configuration

Here we define the file paths for our **5 Data Sources** and the model hyperparameters.

**Data Strategy:**
* **Positives:** FusionAI CSV + 3 External FASTA sources (COSMIC, BLAST validated).
* **Negatives:** FusionAI CSV + 2 Synthetic FASTA sources (Canonical UniProt + Hard Synthetic).
* **Augmentation:** We will apply Reverse Complement augmentation **only to the Negative set** to balance the classes and improve robustness.

In [ ]:
# ============ FILE PATHS ============ 
# Ensure these files exist in your 'datasets/' folder
POSITIVE_CSV = "datasets/fusion_gene_positive_bp_information_with_class_for_modeling.txt"
NEGATIVE_CSV = "datasets/fusion_gene_negative_bp_information_with_class_for_modeling.txt"
TEST_CSV = "datasets/fusion_gene_positive_bp_information_with_class_for_testing.txt"

# Extra Data Sources
EXTRA_POS_FASTA = "datasets/blast_validated_chimeras.fasta,datasets/cosmic_high_confidence_sequences.fna,datasets/chimeras_43466.fa"
NEG_FASTA_CANONICAL = "datasets/false_negative_candidates.fasta"  # Real Human Transcripts
NEG_FASTA_SYNTHETIC = "datasets/false_positive_candidates.fasta"  # Shuffled/Reversed/RandomPairs

COLUMNS = ["Hgene","Hchr","Hbp","Hstrand","Tgene","Tchr","Tbp","Tstrand","5'-gene sequence (10Kb)","3'-gene sequence (10Kb)"]
OUTPUT_DIR = "./hyenadna_breakpoint_model"

# ============ MODEL CONFIGURATION ============
MODEL_NAME = "LongSafari/hyenadna-small-32k-seqlen-hf" 
MAX_LENGTH = 20480  # 20kb Context Window

# ============ TRAINING HYPERPARAMETERS ============
NUM_EPOCHS = 3
BATCH_SIZE = 8          
LEARNING_RATE = 1e-5    
WARMUP_STEPS = 1000      
WEIGHT_DECAY = 0.01     
VAL_SPLIT = 0.2         
SEED = 42

print(f"Model: {MODEL_NAME} | Context: {MAX_LENGTH}bp")

## 3. Advanced Data Engineering

The `DataPreparator` class implements our **Domain Integration Strategy**:
1.  **Multi-Source Loading:** Reads from CSVs and raw FASTA.
2.  **Targeted Augmentation:** Applies Reverse Complement (`_get_reverse_complement`) specifically to the Negative class to prevent class imbalance and teach directionality.
3.  **Strict Deduplication:** Removes duplicate sequences to prevent data leakage.

In [ ]:
class DataPreparator:
    """
    Prepares FuseLens data, implementing the Hard-Negative strategy and Augmentation.
    """
    COLUMNS = ["Hgene","Hchr","Hbp","Hstrand","Tgene","Tchr","Tbp","Tstrand","5'-gene sequence (10Kb)","3'-gene sequence (10Kb)"]

    def __init__(self, positive_csv_path: str, negative_csv_path: str, 
                 extra_fasta_paths: str = "", neg_fasta_canonical_path: str = "", 
                 neg_fasta_synthetic_path: str = ""):
        
        self.positive_csv_path = positive_csv_path
        self.negative_csv_path = negative_csv_path
        self.neg_fasta_canonical_path = neg_fasta_canonical_path
        self.neg_fasta_synthetic_path = neg_fasta_synthetic_path
        
        if extra_fasta_paths:
            self.extra_fasta_paths = [path.strip() for path in extra_fasta_paths.split(',') if path.strip()]
        else:
            self.extra_fasta_paths = []
            
        self.trans_table = str.maketrans("ATCGN", "TAGCN")

    def _get_reverse_complement(self, sequence: str) -> str:
        return sequence.upper().translate(self.trans_table)[::-1]

    def _load_fasta_multiline(self, fasta_path: str) -> List[str]:
        sequences = []
        current_seq_lines = []
        skip_next_sequence = False

        try:
            with open(fasta_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    
                    if line.startswith(">"):
                        if current_seq_lines and not skip_next_sequence:
                            sequences.append("".join(current_seq_lines))
                        current_seq_lines = []
                        skip_next_sequence = "INVALID" in line.upper()

                    elif line.upper() == "SEQUENCE_NOT_AVAILABLE":
                        skip_next_sequence = True
                    else:
                        if not skip_next_sequence:
                            current_seq_lines.append(line)

            if current_seq_lines and not skip_next_sequence:
                sequences.append("".join(current_seq_lines))
                
        except FileNotFoundError:
            print(f" ❌ Error: File not found at {fasta_path}. Skipping.")
        return sequences

    def load_and_prepare_data(self) -> pd.DataFrame:
        # --- 1. POSITIVE DATA ---
        print("--- Loading POSITIVE Data (Label 1) ---")
        positive_csv_df = pd.read_csv(self.positive_csv_path, header=None, names=self.COLUMNS, sep='\t')
        positive_csv_df['sequence'] = positive_csv_df["5'-gene sequence (10Kb)"] + positive_csv_df["3'-gene sequence (10Kb)"]
        positive_csv_df['label'] = 1
        
        all_extra_pos_sequences = []
        for fasta_path in self.extra_fasta_paths:
            print(f"Loading: {fasta_path}")
            all_extra_pos_sequences.extend(self._load_fasta_multiline(fasta_path))
        
        all_positives = pd.concat([
            positive_csv_df[['sequence', 'label']], 
            pd.DataFrame({"sequence": all_extra_pos_sequences, "label": 1})
        ], ignore_index=True)
        print(f"Total Positives: {len(all_positives)}")

        # --- 2. NEGATIVE DATA ---
        print("\n--- Loading NEGATIVE Data (Label 0) ---")
        negative_csv_df = pd.read_csv(self.negative_csv_path, header=None, names=self.COLUMNS, sep='\t')
        negative_csv_df['sequence'] = negative_csv_df["5'-gene sequence (10Kb)"] + negative_csv_df["3'-gene sequence (10Kb)"]
        negative_csv_df['label'] = 0
        
        neg_canonical = self._load_fasta_multiline(self.neg_fasta_canonical_path)
        neg_synthetic = self._load_fasta_multiline(self.neg_fasta_synthetic_path)
        
        all_negatives = pd.concat([
            negative_csv_df[['sequence', 'label']],
            pd.DataFrame({"sequence": neg_canonical, "label": 0}),
            pd.DataFrame({"sequence": neg_synthetic, "label": 0})
        ], ignore_index=True)
        print(f"Total Initial Negatives: {len(all_negatives)}")

        # --- 3. AUGMENTATION & MERGE ---
        print(f"Applying Reverse Complement Augmentation to Negatives...")
        augmented_neg = all_negatives.copy()
        augmented_neg['sequence'] = augmented_neg['sequence'].apply(self._get_reverse_complement)
        all_negatives = pd.concat([all_negatives, augmented_neg], ignore_index=True)

        print("Merging and Deduplicating...")
        combined_df = pd.concat([all_positives, all_negatives], ignore_index=True)
        
        initial_count = len(combined_df)
        combined_df.drop_duplicates(subset=['sequence'], keep='first', inplace=True) 
        print(f"🧹 Removed {initial_count - len(combined_df)} duplicates.")

        # Shuffle
        combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

        print(f"\nDataset prepared: {len(combined_df)} samples")
        return combined_df

print("✅ DataPreparator defined.")

## 4. FuseLens Model Architecture

We customize `HyenaDNA` with a specialized **Attention Pooling Head**.

**Architecture Innovations:**
1.  **Backbone:** `HyenaDNAModel` (Pretrained) extracts long-range features.
2.  **Attention Head:** A learnable layer that assigns a weight $\alpha_t$ to every nucleotide $t$. 
    * *Classification:* Weighted sum of features $\rightarrow$ Probability.
    * *Localization:* `argmax(attention)` $\rightarrow$ Breakpoint Coordinate.
3.  **Forward Logic:** Includes fixes for the `attention_mask` bug found during development.

In [ ]:
class DNABreakpointDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_length):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self): return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx].upper()
        label = self.labels[idx]
        encoding = self.tokenizer(seq, truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        
        input_ids = encoding['input_ids'].squeeze(0)
        if 'attention_mask' in encoding:
            attention_mask = encoding['attention_mask'].squeeze(0)
        else:
            pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id else 2
            attention_mask = (input_ids != pad_id).long()
        
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': torch.tensor(label, dtype=torch.long)}

class HyenaDNAClassifier(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 2):
        super(HyenaDNAClassifier, self).__init__()
        print(f"Loading HyenaDNA: {model_name}")
        torch_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
        
        self.hyenadna = AutoModel.from_pretrained(model_name, trust_remote_code=True, torch_dtype=torch_dtype)
        self.hidden_size = self.hyenadna.config.d_model
        
        # Attention Pooling Head
        self.attention_weights = nn.Linear(self.hidden_size, 1)
        self.classifier = nn.Sequential(
            nn.Linear(self.hidden_size, 256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_labels)
        ).to(torch.float32)

    def forward(self, input_ids, attention_mask=None, labels=None, output_attentions=False):
        # Pass ONLY input_ids to backbone (Fixes TypeError)
        outputs = self.hyenadna(input_ids) 
        
        if hasattr(outputs, 'last_hidden_state'):
            sequence_output = outputs.last_hidden_state.to(torch.float32)
        else:
            sequence_output = outputs[0].to(torch.float32)

        # Apply attention mask manually for pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1)
            sequence_output = sequence_output * mask
        
        # --- ATTENTION MECHANISM ---
        attn_scores = self.attention_weights(sequence_output)
        if attention_mask is not None:
            attn_mask = (attention_mask == 0).unsqueeze(-1)
            attn_scores = attn_scores.masked_fill(attn_mask, float('-inf'))
        
        attn_probs = torch.softmax(attn_scores, dim=1)
        pooled_output = torch.sum(sequence_output * attn_probs, dim=1)
        
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=0.1)
            loss = loss_fct(logits, labels)
            
        return_dict = {'loss': loss, 'logits': logits}
        
        # Localization Output
        if output_attentions:
            return_dict['breakpoint_index'] = torch.argmax(attn_probs, dim=1).squeeze(-1)
            
        return return_dict

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary', pos_label=1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
    auc_score = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else 0.0
    return {'accuracy': accuracy_score(labels, predictions), 'precision': precision, 'recall': recall, 'f1': f1, 'auc': auc_score}

print("✅ Model and Dataset classes defined.")

## 5. Training & Inference Pipeline

The `HyenaDNATrainingPipeline` manages the lifecycle of the model.

**Crucial Fixes Implemented:**
1.  **Saving Fix:** `save_strategy="no"` and manual `torch.save` are used to bypass the `safetensors` shared-memory bug common with Hyena models.
2.  **Prediction Logic:** The `predict` method explicitly passes the `attention_mask` and requests `output_attentions` to enable breakpoint localization.

In [ ]:
class HyenaDNATrainingPipeline:
    def __init__(self, model_name, output_dir, max_length):
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = None
    
    def prepare_datasets(self, train_df, val_split=0.2):
        train_seqs, val_seqs, train_lbls, val_lbls = train_test_split(
            train_df['sequence'].tolist(), train_df['label'].tolist(),
            test_size=val_split, random_state=42, stratify=train_df['label']
        )
        return DNABreakpointDataset(train_seqs, train_lbls, self.tokenizer, self.max_length), \
               DNABreakpointDataset(val_seqs, val_lbls, self.tokenizer, self.max_length)
    
    def train(self, train_dataset, val_dataset, num_epochs, batch_size, learning_rate, warmup_steps, weight_decay):
        self.model = HyenaDNAClassifier(self.model_name, num_labels=2)
        self.model.to(self.device)
        use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        
        training_args = TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            learning_rate=learning_rate,
            warmup_steps=warmup_steps,
            weight_decay=weight_decay,
            lr_scheduler_type="cosine",
            bf16=use_bf16,
            fp16=False if use_bf16 else torch.cuda.is_available(),
            logging_steps=50,
            eval_strategy="epoch",
            save_strategy="no", # Disable auto-save to fix safetensors bug
            save_safetensors=False,
            seed=42
        )
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics
        )
        
        trainer.train()
        
        print("\nSaving final model (Manual Torch Save)...")
        final_model_dir = f"{self.output_dir}/final_model"
        os.makedirs(final_model_dir, exist_ok=True)
        torch.save(self.model.state_dict(), f"{final_model_dir}/pytorch_model.bin")
        self.tokenizer.save_pretrained(final_model_dir)
        print(f"Model saved to {final_model_dir}")
        return trainer
    
    def predict(self, test_csv_path, model_path=None, output_path="predictions.csv", batch_size=8):
        if model_path is None: model_path = f"{self.output_dir}/final_model"
        
        if self.model is None:
            print(f"Loading model from {model_path}...")
            self.model = HyenaDNAClassifier(self.model_name, num_labels=2)
            try:
                load_model(self.model, f"{model_path}/model.safetensors")
            except:
                self.model.load_state_dict(torch.load(f"{model_path}/pytorch_model.bin"))
            self.model.to(self.device)
            self.tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        
        self.model.eval()
        test_df = pd.read_csv(test_csv_path, header=None, names=COLUMNS, sep='\t')
        test_df['sequence'] = test_df["5'-gene sequence (10Kb)"] + test_df["3'-gene sequence (10Kb)"]
        
        # --- Logic for Labels ---
        try:
            test_labels = test_df['label'].tolist()
        except KeyError:
            test_labels = [1] * len(test_df) # Assume all positive if not specified
            
        dataset = DNABreakpointDataset(test_df['sequence'].tolist(), test_labels, self.tokenizer, self.max_length)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        
        all_preds, all_probs, all_breakpoints = [], [], []
        
        with torch.no_grad():
            for batch in tqdm(loader, desc="Predicting"):
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                
                outputs = self.model(input_ids, attention_mask=attention_mask, output_attentions=True)
                
                probs = torch.softmax(outputs['logits'], dim=-1)
                preds = torch.argmax(probs, dim=-1)
                bp_indices = outputs['breakpoint_index'].cpu().numpy()
                
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs[:, 1].cpu().numpy())
                all_breakpoints.extend(bp_indices)
        
        test_df['label'] = test_labels
        test_df['predicted_label'] = all_preds
        test_df['breakpoint_probability'] = all_probs
        test_df['predicted_breakpoint_loc'] = all_breakpoints
        test_df['prediction'] = test_df['predicted_label'].map({0: 'Negative', 1: 'Positive'})
        
        test_df.to_csv(output_path, index=False)
        return test_df

print("✅ Pipeline class defined.")

## 6. Execution: Load, Train, and Evaluate

In [ ]:
# 1. Load Data
data_prep = DataPreparator(
    POSITIVE_CSV, NEGATIVE_CSV, 
    extra_fasta_paths=EXTRA_POS_FASTA,
    neg_fasta_canonical_path=NEG_FASTA_CANONICAL,
    neg_fasta_synthetic_path=NEG_FASTA_SYNTHETIC
)
train_df = data_prep.load_and_prepare_data()

# 2. Initialize Pipeline
pipeline = HyenaDNATrainingPipeline(MODEL_NAME, OUTPUT_DIR, MAX_LENGTH)
train_ds, val_ds = pipeline.prepare_datasets(train_df, val_split=VAL_SPLIT)

# 3. Train
trainer = pipeline.train(train_ds, val_ds, NUM_EPOCHS, BATCH_SIZE, LEARNING_RATE, WARMUP_STEPS, WEIGHT_DECAY)

# 4. View Results
history = trainer.state.log_history
val_metrics = [entry for entry in history if 'eval_accuracy' in entry]
if val_metrics:
    best = val_metrics[-1]
    print("\n--- BEST VALIDATION METRICS ---")
    print(f"Accuracy: {best['eval_accuracy']:.4f} | Recall: {best['eval_recall']:.4f} | AUC: {best['eval_auc']:.4f}")

## 7. Prediction & Visualization

In [ ]:
# 5. Predict on Test Set
predictions_df = pipeline.predict(TEST_CSV, output_path="breakpoint_predictions.csv", batch_size=BATCH_SIZE)

# 6. Visualization
if 'label' not in predictions_df.columns:
    predictions_df['label'] = 1 # Assume positive for plotting if missing

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Probability Dist
sns.histplot(data=predictions_df, x='breakpoint_probability', hue='label', bins=30, kde=True, ax=axes[0], palette=['red', 'green'])
axes[0].set_title("Confidence Distribution")

# ROC Curve
fpr, tpr, _ = roc_curve(predictions_df['label'], predictions_df['breakpoint_probability'])
axes[1].plot(fpr, tpr, color='blue', lw=2, label=f"AUC = {auc(fpr, tpr):.4f}")
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_title("ROC Curve")
axes[1].legend()

# Breakpoint Localization (Histogram of predicted indices)
sns.histplot(data=predictions_df[predictions_df['predicted_label']==1], x='predicted_breakpoint_loc', bins=50, ax=axes[2], color='purple')
axes[2].set_title("Predicted Breakpoint Locations (Positives)")
axes[2].set_xlabel("Genomic Index (0-20480)")
axes[2].axvline(10240, color='orange', linestyle='--', label='Center (Expected)')
axes[2].legend()

plt.tight_layout()
plt.show()